# COSC 2671/ 3047 — Assignment 2 | Group 8
## Notebook 1: Data Extraction and Cleaning

Collects data from the Mastodon public REST API across 5 instances and 13 
migration-related hashtags, then cleans and preprocesses the raw data for analysis.

**Part 1 — Data Collection:** Fetches toots, user metadata, reply edges and boost 
edges. Saves raw output to mastodon_data/

**Part 2 — Data Cleaning:** Removes bots, duplicates, self-loops and out-of-range 
dates. Creates English-only subset. Assigns temporal snapshot labels. 
Saves cleaned output to mastodon_data_clean/

**Output Folders:
- mastodon_data/
- mastodon_data_clean/

**Run this first before any other notebook.**

## SCRAPING DATA ##

In [1]:
import requests
import pandas as pd
import time
import os
from html.parser import HTMLParser
import warnings
warnings.filterwarnings("ignore")

In [2]:
INSTANCES  = ["mastodon.social", "aus.social","fosstodon.org", 
               "infosec.exchange", "scholar.social"]
HASHTAGS   = ["TwitterMigration", "TwitterExodus", "LeavingTwitter", 
               "GoodbyeTwitter", "Mastodon", "FediVerse", "ElonMusk", "birdsite","TwitterRefugee","RIPTwitter","DeleteTwitter"
             ,"TwitterTakeover","NewToMastodon"]
MAX_TOOTS  = 200
DELAY      = 1.5
START_DATE = "2022-10-01"    # Musk acquisition Oct 27 2022 — small buffer before
OUTPUT_DIR = "./mastodon_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
class HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text = []
    def handle_data(self, d):
        self.text.append(d)
    def get_text(self):
        return " ".join(self.text).strip()

def strip_html(html):
    if not html: return ""
    p = HTMLStripper(); p.feed(html); return p.get_text()

def get_domain(url):
    try: return url.split("/")[2]
    except: return "unknown"

def safe_get(url, params=None):
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200: return r.json()
            elif r.status_code == 429: 
                print("  Rate limited — waiting 30s...")
                time.sleep(30)
            else: return []
        except Exception as e:
            print(f"  Error (attempt {attempt+1}): {e}")
            time.sleep(5)
    return []

def fetch_toots(instance, hashtag, max_toots=200, start_date=None):
    """
    Fetch toots with pagination. Stops early if toots go 
    older than start_date to avoid collecting irrelevant data.
    """
    url       = f"https://{instance}/api/v1/timelines/tag/{hashtag}"
    all_toots = []
    max_id    = None

    while len(all_toots) < max_toots:
        params = {"limit": 40}
        if max_id: params["max_id"] = max_id

        batch = safe_get(url, params)
        time.sleep(DELAY)

        if not batch: break

        # Filter batch to only toots after start_date
        if start_date:
            batch = [t for t in batch if t["created_at"] >= start_date]

        all_toots.extend(batch)
        
        # If filtering removed everything, we've gone too far back — stop
        if not batch: break

        max_id = all_toots[-1]["id"]
        if len(batch) < 40: break

    return all_toots[:max_toots]

In [4]:
instance_rows = []

for instance in INSTANCES:
    print(f"Fetching instance info: {instance}")
    data = safe_get(f"https://{instance}/api/v2/instance")
    time.sleep(DELAY)
    if data:
        instance_rows.append({
            "instance":    instance,
            "title":       data.get("title", ""),
            "description": strip_html(data.get("description", "")),
            "user_count":  data.get("usage", {}).get("users", {}).get("active_month", "N/A"),
            "language":    (data.get("languages") or ["unknown"])[0],
        })
        print(f"   Active users/month: {instance_rows[-1]['user_count']}")
    else:
        print(f"   Could not fetch")

df_instances = pd.DataFrame(instance_rows)
df_instances

Fetching instance info: mastodon.social
   Active users/month: 282606
Fetching instance info: aus.social
   Active users/month: 1860
Fetching instance info: fosstodon.org
   Active users/month: 7771
Fetching instance info: infosec.exchange
   Active users/month: 10929
Fetching instance info: scholar.social
   Active users/month: 301


,instance,title,description,user_count,language
0,mastodon.social,Mastodon,"The original server of Mastodon, operated by M...",282606,en
1,aus.social,Aus.Social,Welcome to thundertoot! A Mastodon Instance fo...,1860,en
2,fosstodon.org,Fosstodon,Fosstodon is an invite only Mastodon instance ...,7771,en
3,infosec.exchange,Infosec Exchange,A Mastodon instance for info/cyber security-mi...,10929,en
4,scholar.social,Scholar Social,"Microblogging for researchers, grad students, ...",301,en


In [5]:
all_raw = {}   # keyed by toot_id to deduplicate across instances/hashtags

for instance in INSTANCES:
    for hashtag in HASHTAGS:
        print(f"#{hashtag} @ {instance}...")
        try:
            toots = fetch_toots(instance, hashtag, MAX_TOOTS, START_DATE)
            for t in toots:
                all_raw[t["id"]] = (t, instance, hashtag)
            print(f"  → {len(toots)} toots (unique total: {len(all_raw)})")
        except Exception as e:
            print(f"   Error: {e}")

print(f"\nTotal unique toots collected: {len(all_raw)}")

#TwitterMigration @ mastodon.social...
  → 200 toots (unique total: 200)
#TwitterExodus @ mastodon.social...
  → 200 toots (unique total: 398)
#LeavingTwitter @ mastodon.social...
  → 190 toots (unique total: 588)
#GoodbyeTwitter @ mastodon.social...
  → 200 toots (unique total: 786)
#Mastodon @ mastodon.social...
  → 200 toots (unique total: 982)
#FediVerse @ mastodon.social...
  → 200 toots (unique total: 1146)
#ElonMusk @ mastodon.social...
  → 200 toots (unique total: 1346)
#birdsite @ mastodon.social...
  → 200 toots (unique total: 1543)
#TwitterRefugee @ mastodon.social...
  → 200 toots (unique total: 1740)
#RIPTwitter @ mastodon.social...
  → 200 toots (unique total: 1939)
#DeleteTwitter @ mastodon.social...
  → 200 toots (unique total: 2138)
#TwitterTakeover @ mastodon.social...
  → 200 toots (unique total: 2335)
#NewToMastodon @ mastodon.social...
  → 200 toots (unique total: 2532)
#TwitterMigration @ aus.social...
  → 200 toots (unique total: 2732)
#TwitterExodus @ aus.social

In [6]:
# Verify no pre-2022 data slipped through
dates = pd.Series([t["created_at"][:7] for t, _, _ in all_raw.values()])
print("Toot counts by month:")
print(dates.value_counts().sort_index().to_string())

Toot counts by month:
2022-10      33
2022-11     349
2022-12     362
2023-01      85
2023-02      81
2023-03     119
2023-04     236
2023-05      92
2023-06     113
2023-07     660
2023-08     320
2023-09     245
2023-10     171
2023-11      94
2023-12      89
2024-01      28
2024-02      30
2024-03      40
2024-04      38
2024-05      44
2024-06      61
2024-07      45
2024-08     135
2024-09      70
2024-10     183
2024-11     492
2024-12      69
2025-01     207
2025-02      92
2025-03      94
2025-04      34
2025-05      57
2025-06      20
2025-07      37
2025-08      21
2025-09       6
2025-10       9
2025-11      19
2025-12      12
2026-01      72
2026-02      22
2026-03     214
2026-04     244
2026-05    1848


In [7]:
# Creating of column names and columns for required output files
toot_rows, user_dict, reply_rows, boost_rows = [], {}, [], []

for toot_id, (toot, src_instance, hashtag) in all_raw.items():
    is_boost = toot.get("reblog") is not None
    orig     = toot["reblog"] if is_boost else toot
    acc      = toot["account"]
    orig_acc = orig["account"]

    toot_rows.append({
        "toot_id":                toot["id"],
        "original_toot_id":       orig["id"],
        "username":               acc["username"],
        "user_id":                acc["id"],
        "user_instance":          get_domain(acc["url"]),
        "user_followers":         acc["followers_count"],
        "user_following":         acc["following_count"],
        "user_posts_total":       acc["statuses_count"],
        "user_joined":            acc["created_at"],
        "content_text":           strip_html(orig.get("content", "")),
        "language":               orig.get("language", ""),
        "hashtags":               ",".join([h["name"] for h in orig.get("tags", [])]),
        "reblogs_count":          orig.get("reblogs_count", 0),
        "favourites_count":       orig.get("favourites_count", 0),
        "replies_count":          orig.get("replies_count", 0),
        "in_reply_to_id":         toot.get("in_reply_to_id"),
        "in_reply_to_account_id": toot.get("in_reply_to_account_id"),
        "is_boost":               is_boost,
        "boosted_from_instance":  get_domain(orig_acc["url"]) if is_boost else None,
        "boosted_user_id":        orig_acc["id"] if is_boost else None,
        "created_at":             toot["created_at"],
        "source_instance":        src_instance,
        "hashtag_searched":       hashtag,
    })

    if acc["id"] not in user_dict:
        user_dict[acc["id"]] = {
            "user_id":     acc["id"],
            "username":    acc["username"],
            "instance":    get_domain(acc["url"]),
            "followers":   acc["followers_count"],
            "following":   acc["following_count"],
            "total_posts": acc["statuses_count"],
            "joined_at":   acc["created_at"],
            "bot":         acc.get("bot", False),
            "bio":         strip_html(acc.get("note", "")),
        }

    if toot.get("in_reply_to_account_id"):
        reply_rows.append({
            "source_user_id":    acc["id"],
            "source_username":   acc["username"],
            "source_instance":   get_domain(acc["url"]),
            "target_account_id": toot["in_reply_to_account_id"],
            "toot_id":           toot["id"],
            "reply_to_toot_id":  toot["in_reply_to_id"],
            "created_at":        toot["created_at"],
            "edge_type":         "reply"
        })

    if is_boost:
        boost_rows.append({
            "source_user_id":   acc["id"],
            "source_username":  acc["username"],
            "source_instance":  get_domain(acc["url"]),
            "target_user_id":   orig_acc["id"],
            "target_username":  orig_acc["username"],
            "target_instance":  get_domain(orig_acc["url"]),
            "toot_id":          toot["id"],
            "original_toot_id": orig["id"],
            "created_at":       toot["created_at"],
            "edge_type":        "boost"
        })

df_toots = pd.DataFrame(toot_rows).drop_duplicates(subset="toot_id")
df_users  = pd.DataFrame(list(user_dict.values()))
df_reply  = pd.DataFrame(reply_rows)
df_boost  = pd.DataFrame(boost_rows)

print(f"Toots:        {len(df_toots)}")
print(f"Users:        {len(df_users)}")
print(f"Reply edges:  {len(df_reply)}")
print(f"Boost edges:  {len(df_boost)}")

Toots:        7292
Users:        3410
Reply edges:  840
Boost edges:  0


In [8]:
# Boost edges need a separate endpoint — get who boosted each toot
high_boost_toots = df_toots[df_toots["reblogs_count"] > 0].copy()
print(f"Toots that were boosted by others: {len(high_boost_toots)}")

boost_edge_rows = []

# Querying top 50 most boosted toots only (to keep it manageable)
top_boosted = high_boost_toots.nlargest(50, "reblogs_count")

for _, row in top_boosted.iterrows():
    src_instance = row["source_instance"] if "source_instance" in row else "mastodon.social"
    url = f"https://{src_instance}/api/v1/statuses/{row['toot_id']}/reblogged_by"
    boosters = safe_get(url)
    time.sleep(DELAY)

    for booster in boosters:
        boost_edge_rows.append({
            "source_user_id":   booster["id"],
            "source_username":  booster["username"],
            "source_instance":  get_domain(booster["url"]),
            "target_user_id":   row["user_id"],
            "target_username":  row["username"],
            "target_instance":  row["user_instance"],
            "toot_id":          row["toot_id"],
            "reblogs_count":    row["reblogs_count"],
            "created_at":       row["created_at"],
            "edge_type":        "boost"
        })

df_boost = pd.DataFrame(boost_edge_rows)
print(f"Boost edges collected: {len(df_boost)}")

Toots that were boosted by others: 3952
Boost edges collected: 2000


In [9]:
print("=== SAMPLE TOOTS ===")
print(df_toots[["username", "user_instance", "created_at", "content_text"]].head(5).to_string())

print("\n=== DATE RANGE ===")
print(f"Earliest: {df_toots['created_at'].min()}")
print(f"Latest:   {df_toots['created_at'].max()}")

print("\n=== TOP INSTANCES IN DATA ===")
print(df_toots["user_instance"].value_counts().head(10))

print("\n=== HASHTAG BREAKDOWN ===")
print(df_toots["hashtag_searched"].value_counts())

print("\n=== LANGUAGES ===")
print(df_toots["language"].value_counts().head(5))

print("\n=== REPLY vs BOOST EDGES ===")
print(f"Reply edges: {len(df_reply)}")
print(f"Boost edges: {len(df_boost)}")

=== SAMPLE TOOTS ===
               username   user_instance                created_at                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            content_text
0                 JdeBP  mastodonapp.uk  2026-05-24T23:15:19.000Z  @ john   With 1 exception, every 'no' on that list is a person who has not themselves abandoned Twitter and publicly lists their Twitter accounts, sometimes even as the only social media accounts th

In [10]:
df_toots.to_csv(    f"{OUTPUT_DIR}/toots.csv",        index=False)
df_users.to_csv(    f"{OUTPUT_DIR}/users.csv",         index=False)
df_reply.to_csv(    f"{OUTPUT_DIR}/edges_reply.csv",   index=False)
df_boost.to_csv(    f"{OUTPUT_DIR}/edges_boost.csv",   index=False)
df_instances.to_csv(f"{OUTPUT_DIR}/instances.csv",     index=False)

print(f"\n  Saved to: {os.path.abspath(OUTPUT_DIR)}/")


  Saved to: C:\Users\LENOVO\Social Media and Network Analysis\Assignment 2\mastodon_data/


In [11]:
# Checking which hashtags are driving the toots
may = df_toots[df_toots["created_at"].str[:7] == "2026-05"]
print(may["hashtag_searched"].value_counts())

hashtag_searched
FediVerse           600
ElonMusk            598
Mastodon            467
TwitterMigration    166
birdsite             14
DeleteTwitter         3
Name: count, dtype: int64


## DATA CLEANING ##

In [12]:
# Load all files
df_toots     = pd.read_csv("mastodon_data/toots.csv")
df_users     = pd.read_csv("mastodon_data/users.csv")
df_reply     = pd.read_csv("mastodon_data/edges_reply.csv")
df_boost     = pd.read_csv("mastodon_data/edges_boost.csv")
df_instances = pd.read_csv("mastodon_data/instances.csv")

print(f"Loaded:")
print(f"  toots:     {len(df_toots)}")
print(f"  users:     {len(df_users)}")
print(f"  replies:   {len(df_reply)}")
print(f"  boosts:    {len(df_boost)}")
print(f"  instances: {len(df_instances)}")

Loaded:
  toots:     7292
  users:     3410
  replies:   840
  boosts:    2000
  instances: 5


In [13]:
## Clean Toots ##

print(f"Before cleaning: {len(df_toots)} toots")

# 1. Drop null/empty content
df_toots_clean = df_toots[df_toots["content_text"].fillna("").str.strip() != ""]

# 2. Cap date range — remove May 2026 spike
df_toots_clean = df_toots_clean[df_toots_clean["created_at"] <= "2026-04-30"]

# 3. Add year_month column for temporal analysis
df_toots_clean["year_month"] = df_toots_clean["created_at"].str[:7]

# 4. Add snapshot column — core of your temporal analysis
def assign_snapshot(ym):
    if "2022-10" <= ym <= "2022-12":
        return "Snapshot 1 — Peak Exodus (Oct–Dec 2022)"
    elif "2023-01" <= ym <= "2023-12":
        return "Snapshot 2 — Post Settling (2023)"
    elif "2024-01" <= ym <= "2026-04":
        return "Snapshot 3 — Current State (2024–2026)"
    else:
        return "Out of range"

df_toots_clean["snapshot"] = df_toots_clean["year_month"].apply(assign_snapshot)

print(f"After cleaning: {len(df_toots_clean)} toots")
print(f"\nSnapshot distribution:")
print(df_toots_clean["snapshot"].value_counts().to_string())

Before cleaning: 7292 toots
After cleaning: 5436 toots

Snapshot distribution:
snapshot
Snapshot 3 — Current State (2024–2026)     2387
Snapshot 2 — Post Settling (2023)          2305
Snapshot 1 — Peak Exodus (Oct–Dec 2022)     744


In [14]:
## Clean Users ##
print(f"Before cleaning: {len(df_users)} users")

# 1. Remove bot accounts
df_users_clean = df_users[df_users["bot"] == False]

# 2. Remove unrealistic join years (Mastodon launched 2016)
df_users_clean["join_year"] = df_users_clean["joined_at"].str[:4].astype(int)
df_users_clean = df_users_clean[df_users_clean["join_year"].between(2016, 2026)]

# 3. Add migrant flag — joined during exodus window
df_users_clean["is_migrant"] = df_users_clean["join_year"] == 2022

print(f"After cleaning: {len(df_users_clean)} users")
print(f"\nUsers by join year:")
print(df_users_clean["join_year"].value_counts().sort_index().to_string())
print(f"\nMigrants (joined 2022): {df_users_clean['is_migrant'].sum()}")
print(f"Pre-existing users:     {(~df_users_clean['is_migrant']).sum()}")

Before cleaning: 3410 users
After cleaning: 3298 users

Users by join year:
join_year
2016      13
2017      76
2018     100
2019      68
2020      72
2021      65
2022    1556
2023     634
2024     252
2025     249
2026     213

Migrants (joined 2022): 1556
Pre-existing users:     1742


In [15]:
## Filtering English only ##
# Keep full dataset for network analysis

# English-only subset for NLP
df_toots_en = df_toots[df_toots["language"] == "en"].copy()

print(f"All toots (for network):    {len(df_toots)}")
print(f"English toots (for NLP):    {len(df_toots_en)}")
print(f"\nLanguage breakdown (full):")
print(df_toots["language"].value_counts().head(8).to_string())

All toots (for network):    7292
English toots (for NLP):    5604

Language breakdown (full):
language
en    5604
de     829
fr      99
es      86
nl      52
ja      49
pl      45
ar      34


In [16]:
## Cleaning Reply Edges ##
print(f"Before cleaning: {len(df_reply)} reply edges")

# 1. Remove self-loops (user replying to themselves)
df_reply_clean = df_reply[
    df_reply["source_user_id"].astype(str) != df_reply["target_account_id"].astype(str)
]

# 2. Only keep edges where source user is in clean users list
valid_users = set(df_users_clean["user_id"].astype(str))
df_reply_clean = df_reply_clean[df_reply_clean["source_user_id"].astype(str).isin(valid_users)]

# 3. Add snapshot label
df_reply_clean["snapshot"] = df_reply_clean["created_at"].str[:7].apply(assign_snapshot)

# 4. Remove out of range
df_reply_clean = df_reply_clean[df_reply_clean["snapshot"] != "Out of range"]
df_reply_clean = df_reply_clean[df_reply_clean["created_at"] <= "2026-04-30"]

print(f"After cleaning: {len(df_reply_clean)} reply edges")
print(f"\nBy snapshot:")
print(df_reply_clean["snapshot"].value_counts().to_string())

Before cleaning: 840 reply edges
After cleaning: 493 reply edges

By snapshot:
snapshot
Snapshot 2 — Post Settling (2023)          243
Snapshot 3 — Current State (2024–2026)     191
Snapshot 1 — Peak Exodus (Oct–Dec 2022)     59


In [17]:
# Saving all cleaned files

OUTPUT_DIR = "mastodon_data_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Combine reply + boost into unified edge list
df_reply_unified = df_reply_clean[[
    "source_user_id", "source_username", "source_instance",
    "target_account_id", "created_at", "snapshot", "edge_type"
]].rename(columns={"target_account_id": "target_user_id"})

df_boost_unified = df_boost[[
    "source_user_id", "source_username", "source_instance",
    "target_user_id", "created_at", "edge_type"
]]

# Add snapshot to boost
df_boost_clean = df_boost.copy()
df_boost_clean = df_boost_clean[df_boost_clean["created_at"] <= "2026-04-30"]
df_boost_clean["snapshot"] = df_boost_clean["created_at"].str[:7].apply(assign_snapshot)
df_boost_clean = df_boost_clean[df_boost_clean["snapshot"] != "Out of range"]

df_boost_unified = df_boost_clean[[
    "source_user_id", "source_username", "source_instance",
    "target_user_id", "created_at", "snapshot", "edge_type"
]]

df_edges_all = pd.concat([df_reply_unified, df_boost_unified], ignore_index=True)

# English only toots — filter from cleaned toots
df_toots_en = df_toots_clean[df_toots_clean["language"] == "en"].copy()

# Save all
df_toots_clean.to_csv(  f"{OUTPUT_DIR}/toots_clean.csv",        index=False)
df_toots_en.to_csv(     f"{OUTPUT_DIR}/toots_english.csv",       index=False)
df_users_clean.to_csv(  f"{OUTPUT_DIR}/users_clean.csv",          index=False)
df_reply_clean.to_csv(  f"{OUTPUT_DIR}/edges_reply_clean.csv",    index=False)
df_boost_clean.to_csv(  f"{OUTPUT_DIR}/edges_boost_clean.csv",    index=False)
df_edges_all.to_csv(    f"{OUTPUT_DIR}/edges_all.csv",            index=False)
df_instances.to_csv(    f"{OUTPUT_DIR}/instances.csv",            index=False)

print(f"\n  Saved to: {os.path.abspath(OUTPUT_DIR)}/")


  Saved to: C:\Users\LENOVO\Social Media and Network Analysis\Assignment 2\mastodon_data_clean/
